# 14. 하이퍼파라미터 재탐색 (v2)

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import random
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

RANDOM_STATE = 42


## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('train:', train.shape, '/ test:', test.shape)


train: (2994, 18) / test: (3000, 17)


## 2. 파생변수 생성

In [3]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data


train = add_features(train)
test = add_features(test)
print('파생변수 적용 후 train shape:', train.shape)


파생변수 적용 후 train shape: (2994, 37)


## 3. 인코딩

In [4]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)


x_train: (2994, 35) / x_test: (3000, 35)


## 4. 탐색 공간

In [5]:
random.seed(RANDOM_STATE)
N_TRIALS = 20

search_space = {
    'learning_rate': [0.02, 0.03, 0.04, 0.05, 0.06, 0.08],
    'num_leaves': [31, 63, 95, 127, 160],
    'min_child_samples': [5, 10, 20, 30],
    'reg_alpha': [0.0, 0.05, 0.1, 0.15, 0.2, 0.3],
    'reg_lambda': [0.0, 0.05, 0.1, 0.15, 0.2, 0.3],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}
MAX_ESTIMATORS = 3000

trial_params = []
for _ in range(N_TRIALS):
    params = {k: random.choice(v) for k, v in search_space.items()}
    params['n_estimators'] = MAX_ESTIMATORS
    trial_params.append(params)

baseline_04 = dict(n_estimators=1200, learning_rate=0.08, reg_alpha=0.15, reg_lambda=0.1,
                    subsample=0.8, colsample_bytree=0.8)
baseline_06 = dict(n_estimators=5000, learning_rate=0.03, num_leaves=127, min_child_samples=10)


## 5. CV 탐색

In [6]:
def cv_mae_with_early_stopping(params):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr_idx, va_idx in kf.split(x_train):
        X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        model = LGBMRegressor(**params, random_state=RANDOM_STATE, verbose=-1)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
        maes.append(mean_absolute_error(y_va, model.predict(X_va)))
    return np.mean(maes)


def cv_mae_fixed_iter(params):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr_idx, va_idx in kf.split(x_train):
        X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        model = LGBMRegressor(**params, random_state=RANDOM_STATE, verbose=-1)
        model.fit(X_tr, y_tr)
        maes.append(mean_absolute_error(y_va, model.predict(X_va)))
    return np.mean(maes)


search_results = []

mae_04 = cv_mae_fixed_iter(baseline_04)
search_results.append((mae_04, baseline_04, '04'))
print(f'#04: MAE = {mae_04:.4f}')

mae_06 = cv_mae_with_early_stopping(baseline_06)
search_results.append((mae_06, baseline_06, '06'))
print(f'#06: MAE = {mae_06:.4f}')

for i, params in enumerate(trial_params):
    mae = cv_mae_with_early_stopping(params)
    search_results.append((mae, params, f'trial_{i+1}'))
    print(f'[{i+1}/{N_TRIALS}] MAE = {mae:.4f}  {params}')

search_results.sort(key=lambda x: x[0])
best_mae, best_params, best_tag = search_results[0]
print(f'\nbest: {best_tag}  MAE = {best_mae:.4f}')
print(best_params)


#04: MAE = 0.1815


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

#06: MAE = 0.1740


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

[1/20] MAE = 0.1814  {'learning_rate': 0.08, 'num_leaves': 31, 'min_child_samples': 5, 'reg_alpha': 0.3, 'reg_lambda': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 3000}


...

[18/20] MAE = 0.1794  {'learning_rate': 0.05, 'num_leaves': 160, 'min_child_samples': 30, 'reg_alpha': 0.2, 'reg_lambda': 0.1, 'subsample': 0.7, 'colsample_bytree': 0.7, 'n_estimators': 3000}


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

[19/20] MAE = 0.1806  {'learning_rate': 0.08, 'num_leaves': 160, 'min_child_samples': 20, 'reg_alpha': 0.3, 'reg_lambda': 0.1, 'subsample': 0.7, 'colsample_bytree': 0.9, 'n_estimators': 3000}


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

[20/20] MAE = 0.1743  {'learning_rate': 0.05, 'num_leaves': 63, 'min_child_samples': 30, 'reg_alpha': 0.0, 'reg_lambda': 0.3, 'subsample': 0.9, 'colsample_bytree': 0.8, 'n_estimators': 3000}

best: 06  MAE = 0.1740
{'n_estimators': 5000, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 10}


## 6. 최종 학습 + 제출 파일 저장

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
final_test_pred = np.zeros(len(x_test))
final_oof = np.zeros(len(x_train))

uses_early_stopping = best_tag != '04'

for tr_idx, va_idx in kf.split(x_train):
    X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    model = LGBMRegressor(**best_params, random_state=RANDOM_STATE, verbose=-1)
    if uses_early_stopping:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    else:
        model.fit(X_tr, y_tr)
    final_oof[va_idx] = model.predict(X_va)
    final_test_pred += model.predict(x_test) / kf.n_splits

print(f'최종 CV MAE: {mean_absolute_error(y_train, final_oof):.4f}')

os.makedirs('../submissions', exist_ok=True)
final_pred_14 = np.clip(final_test_pred, 0, 1)
sample_submission['stress_score'] = final_pred_14
sample_submission.to_csv('../submissions/submit_14_hyperparam_search_v2.csv', index=False)
sample_submission.head()


C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\ahssk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

최종 CV MAE: 0.1740


,ID,stress_score
0,TEST_0000,0.547939
1,TEST_0001,0.837801
2,TEST_0002,0.304132
3,TEST_0003,0.488694
4,TEST_0004,0.604630
